# Capacidade Produtiva por Fornecedor e Célula

**Objetivo**: consolidar em uma única tabela, por **fornecedor** (e, quando aplicável, por **célula produtiva**):

- **Média de Alocado (Data Planejada)** — últimos 12 meses
- **Média de Entrega** — últimos 12 meses
- **Capacidade Hoje** — capacidade produtiva mensal vigente, por célula

**Fontes**:
- `insider-data-lake.sop_silver.supply_chain_efficiency_model_input` — alocado (`planned_quantity`) e entrega (`received_quantity`) por fornecedor
- `insider-data-lake.integrated.muninn_apparel_manufacturer_production_units_products` (+ cadeia de joins até `muninn_suppliers`) — capacidade semanal/mensal vigente por célula

⚠️ **Grãos diferentes**: alocado/entrega são por fornecedor; capacidade é por fornecedor × célula. O merge final replica o alocado/entrega do fornecedor em cada linha de célula.

In [13]:
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = 'insider-data-lake'
client = bigquery.Client(project=PROJECT_ID)

def query_to_dataframe(query: str) -> pd.DataFrame:
    return client.query(query).to_dataframe()

/Users/insider/LA_Coding_Projects/.venv-1/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


## 1. Média de Alocado e Entrega por Fornecedor (12 últimos meses)

In [14]:
sql_alocado_entrega = """
WITH base AS (
    SELECT
        supplier_name,
        DATE_TRUNC(dt_planned_entry_warehouse, MONTH) AS mes_planejado,
        DATE_TRUNC(dt_largest_entry_warehouse, MONTH) AS mes_recebido,
        planned_quantity,
        received_quantity
    FROM `insider-data-lake.sop_silver.supply_chain_efficiency_model_input`
    WHERE production_order_type NOT IN ('flexible', 'converted')
      AND cycle_name IS NOT NULL
      AND (supplier_relationship_status IS NULL
           OR supplier_relationship_status NOT IN ('terminated', 'discontinued'))
),
alocado_mensal AS (
    SELECT supplier_name, mes_planejado AS mes, SUM(planned_quantity) AS qtd_alocada
    FROM base
    WHERE mes_planejado >= DATE_TRUNC(DATE_SUB(CURRENT_DATE(), INTERVAL 12 MONTH), MONTH)
      AND mes_planejado < DATE_TRUNC(CURRENT_DATE(), MONTH)
    GROUP BY supplier_name, mes
),
entrega_mensal AS (
    SELECT supplier_name, mes_recebido AS mes, SUM(received_quantity) AS qtd_entregue
    FROM base
    WHERE mes_recebido >= DATE_TRUNC(DATE_SUB(CURRENT_DATE(), INTERVAL 12 MONTH), MONTH)
      AND mes_recebido < DATE_TRUNC(CURRENT_DATE(), MONTH)
    GROUP BY supplier_name, mes
)
SELECT
    COALESCE(a.supplier_name, e.supplier_name) AS supplier_name,
    AVG(a.qtd_alocada) AS media_alocado_12m,
    AVG(e.qtd_entregue) AS media_entrega_12m
FROM alocado_mensal AS a
FULL OUTER JOIN entrega_mensal AS e
    ON a.supplier_name = e.supplier_name AND a.mes = e.mes
GROUP BY supplier_name
ORDER BY media_alocado_12m DESC
"""

df_alocado_entrega = query_to_dataframe(sql_alocado_entrega)
df_alocado_entrega.head(10)

/Users/insider/LA_Coding_Projects/.venv-1/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,supplier_name,media_alocado_12m,media_entrega_12m
0,BAE BRASIL,186447.833333,151356.833333
1,BY COTTON,55989.000000,47761.700000
2,ART LIVRE,43928.916667,39310.583333
3,WARUSKY,38790.600000,19282.666667
4,RIZLLEP,33453.500000,29888.818182
5,MALHAS D'STEFANO,31868.833333,26499.250000
6,LUTESTIL,31667.000000,9405.666667
7,Lunelli Nordeste,26546.428571,28724.500000
8,INTERTEXTIL,21521.909091,7384.090909
9,TOP SPORT,20711.666667,NaN


## 2. Capacidade Produtiva Hoje, por Célula

In [15]:
sql_capacidade_celula = """
SELECT
    s.alias AS supplier_name,
    ampu.apparel_manufacturer_cell_number AS cell_number,
    ampu.label AS cell_label,
    MAX(ampup.weekly_maximum_productive_capacity) AS capacidade_semanal_hoje,
    4 * MAX(ampup.weekly_maximum_productive_capacity) AS capacidade_mensal_hoje
FROM `insider-data-lake.integrated.muninn_apparel_manufacturer_production_units_products` AS ampup
LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturers_products` AS amp
    ON amp.id = ampup.apparel_manufacturer_product_id
LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturer_production_units` AS ampu
    ON ampu.id = ampup.apparel_manufacturer_production_unit_id
LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturers` AS am
    ON am.id = ampu.apparel_manufacturer_id
LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS s
    ON s.id = am.supplier_id
WHERE amp.status IN ('available', 'approved', 'incubation')
  AND ampup.weekly_maximum_productive_capacity > 0
GROUP BY supplier_name, cell_number, cell_label
ORDER BY supplier_name, cell_number
"""

df_capacidade_celula = query_to_dataframe(sql_capacidade_celula)
df_capacidade_celula.head(10)



/Users/insider/LA_Coding_Projects/.venv-1/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,supplier_name,cell_number,cell_label,capacidade_semanal_hoje,capacidade_mensal_hoje
0,ART LIVRE,1,modal/visup 1 - Gola U,7500,30000
1,ARTIGO X,1,bone,2500,10000
2,ASGA BRINDES,1,Brindes,5000,20000
3,AZZURRA,1,Authentico/ Hydrid Jogger,500,2000
4,AZZURRA,2,New York Stretch,1000,4000
5,AZZURRA,3,Shorts endorphine,500,2000
6,BAE BRASIL,1,cueca,12500,50000
7,BAE BRASIL,2,Calcinhas,4500,18000
8,BAE BRASIL,3,Camisetas - modal/visup,37000,148000
9,BAE BRASIL,5,perfect-top,13000,52000


## 3. Tabela Final (aproximada) — Fornecedor × Célula

⚠️ Esta versão replica a **média do fornecedor** em todas as suas células (aproximação). A
versão correta, com alocado real por célula, está na Seção 3c mais abaixo.


In [16]:
df_fornecedor_celula = df_capacidade_celula.merge(
    df_alocado_entrega,
    on="supplier_name",
    how="outer",
)

tabela_final = df_fornecedor_celula.rename(columns={
    "supplier_name": "Fornecedor",
    "cell_label": "Célula",
    "media_alocado_12m": "Média de Alocado (Data Planejada) - 12 ÚLTIMOS MESES",
    "media_entrega_12m": "Média de Entrega - 12 ÚLTIMOS MESES",
    "capacidade_mensal_hoje": "Capacidade Hoje",
})[[
    "Fornecedor",
    "Célula",
    "Média de Alocado (Data Planejada) - 12 ÚLTIMOS MESES",
    "Média de Entrega - 12 ÚLTIMOS MESES",
    "Capacidade Hoje",
]].sort_values("Fornecedor").reset_index(drop=True)

tabela_final

,Fornecedor,Célula,Média de Alocado (Data Planejada) - 12 ÚLTIMOS MESES,Média de Entrega - 12 ÚLTIMOS MESES,Capacidade Hoje
0,ABBA,NaN,11376.166667,7693.000000,<NA>
1,ART LIVRE,modal/visup 1 - Gola U,43928.916667,39310.583333,30000
2,ARTIGO X,bone,4196.818182,3514.833333,10000
3,ASGA BRINDES,Brindes,NaN,NaN,20000
4,AZZURRA,Authentico/ Hydrid Jogger,5484.875000,5768.833333,2000
...,...,...,...,...,...
136,T Christina,Macacão Fitness InSculpt Feminino,4485.500000,2020.000000,700
137,TEXPORT,CAMISETA ESPORTIVA,1162.000000,NaN,4000
138,TOP SPORT,NaN,20711.666667,NaN,<NA>
139,Têxtil WM,Camiseta MC Grafiato,9991.000000,2470.000000,5000


## 3b. Alocado real por Célula — via `capacity_use` (⚠️ superada pela Seção 3d)

⚠️ **Esta abordagem foi superada.** A Seção 3d usa `supply_production_orders` (join por
`op_code`) em vez de `capacity_use`, cobrindo 12 meses completos (aqui o máximo é 9) e trazendo
também a produção/entrega real por célula. Mantida como histórico da investigação.

A Seção 3 acima replica a **média de alocado do fornecedor** em todas as suas células — é uma
aproximação, não a alocação real por célula.

⚠️ **Importante**: "Capacidade Hoje" é o cadastro vigente no Muninn (Seção 2,
`weekly_maximum_productive_capacity`) — **um snapshot do estado atual**, não um histórico.
`capacity_use` não substitui isso; ela serve **apenas para trazer o alocado real por célula**
(`total_quant_planned`), que não existe em nenhuma tabela de cadastro. Por isso esta seção não
recalcula capacidade — só extrai a alocação mensal histórica e faz merge com a capacidade atual
da Seção 2 mais adiante.

**Achados validados no BigQuery (execução real, não suposição):**

1. **Join célula → fornecedor — caminho curto, cobertura 100%:**
   `production_units.apparel_manufacturer_id` → `muninn_apparel_manufacturers.id/supplier_id` →
   `muninn_suppliers.id`. O caminho via `_products` (usado na Seção 2) cobre só 80,8% das
   células — não usar aqui.

2. **`capacity_use` é prospectivo, não histórico.** Cada `dt_reference` (snapshot diário) só
   enxerga 5 meses **para frente** a partir do mês corrente daquele snapshot. Filtrar
   `dt_reference = MAX(dt_reference)` traz só ago–dez/2026 (planejamento futuro), nunca os
   últimos 12 meses reais.

   **Correção aplicada:** para cada `month_dt` alvo no passado, usamos o `dt_reference` cujo mês
   seja igual ao `month_dt` (o snapshot tirado quando aquele mês ainda era o mês corrente) — um
   `dt_reference` por mês, não o mais recente do sistema.

3. **⚠️ A janela de "12 meses" nunca é completa.** Rodando a query, `n_meses_com_dado` variou de
   1 a **9** (nunca 12) — o snapshot mais antigo disponível é de 2025-11-21, o que limita o
   histórico. Mediana de 7 meses; **37 células têm só 2 meses de dado**. Por isso a coluna
   `n_meses_com_dado` é mantida no resultado — sempre checar antes de confiar na média de uma
   célula específica.

4. **Bug de tipo corrigido:** `dt_reference` e `month_dt` são `TIMESTAMP` em `capacity_use`, não
   `DATE`. As comparações usam `DATE(...)` explicitamente para evitar erro de tipo no BigQuery.


In [17]:
sql_alocado_por_celula = """
WITH snapshot_por_mes AS (
    -- Para cada mes historico, pega o dt_reference tirado durante aquele mes
    -- (nao o mais recente do sistema, que so enxerga 5 meses para frente).
    SELECT
        DATE(month_dt) AS month_dt,
        MAX(dt_reference) AS dt_reference_usado
    FROM `insider-data-lake.sop_bronze.capacity_use`
    WHERE DATE_TRUNC(DATE(dt_reference), MONTH) = DATE_TRUNC(DATE(month_dt), MONTH)
      AND DATE(month_dt) >= DATE_TRUNC(DATE_SUB(CURRENT_DATE(), INTERVAL 12 MONTH), MONTH)
      AND DATE(month_dt) < DATE_TRUNC(CURRENT_DATE(), MONTH)
    GROUP BY month_dt
),
alocado_por_celula_mes AS (
    -- Alocado real por celula×mes. Capacidade NAO vem daqui — capacidade e o
    -- cadastro vigente no Muninn (Secao 2), um snapshot do estado atual.
    SELECT
        cu.apparel_manufacturer_production_unit_id,
        DATE(cu.month_dt) AS month_dt,
        cu.total_quant_planned
    FROM `insider-data-lake.sop_bronze.capacity_use` AS cu
    JOIN snapshot_por_mes AS sm
        ON DATE(cu.month_dt) = sm.month_dt
        AND cu.dt_reference = sm.dt_reference_usado
)
SELECT
    s.alias AS supplier_name,
    ampu.apparel_manufacturer_cell_number AS cell_number,
    ampu.label AS cell_label,
    AVG(c.total_quant_planned) AS media_alocado_celula_12m,
    COUNT(DISTINCT c.month_dt) AS n_meses_com_dado
FROM alocado_por_celula_mes AS c
LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturer_production_units` AS ampu
    ON ampu.id = c.apparel_manufacturer_production_unit_id
LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturers` AS am
    ON am.id = ampu.apparel_manufacturer_id
LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS s
    ON s.id = am.supplier_id
GROUP BY supplier_name, cell_number, cell_label
ORDER BY supplier_name, cell_number
"""

df_alocado_por_celula = query_to_dataframe(sql_alocado_por_celula)
df_alocado_por_celula.head(10)



/Users/insider/LA_Coding_Projects/.venv-1/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,supplier_name,cell_number,cell_label,media_alocado_celula_12m,n_meses_com_dado
0,ABBA,1,Polo Core,219.125000,8
1,ABBA,2,Henley Core,0.000000,8
2,ABBA,3,Core T-shirt,331.750000,8
3,ART LIVRE,1,modal/visup 1 - Gola U,9050.000000,9
4,ART LIVRE,2,modal/visup 2 - Gola V,1550.625000,8
5,ART LIVRE,3,modal/visup 3 - regata,1638.833333,6
6,ARTIGO X,1,bone,258.222222,9
7,ASGA BRINDES,1,Brindes,0.000000,1
8,AZZURRA,1,Authentico/ Hydrid Jogger,0.000000,9
9,AZZURRA,2,New York Stretch,0.000000,9


## 3c. Tabela Final (correta) — Fornecedor × Célula

Junta a **capacidade atual** (Seção 2, cadastro vigente no Muninn — snapshot de hoje, não
histórico) com o **alocado real por célula** (Seção 3b, `capacity_use` histórico). Diferente da
Seção 3, aqui a alocação não é aproximada pelo total do fornecedor — é o valor real de cada
célula.

⚠️ "Entrega" não está disponível neste grão — `capacity_use`/Muninn não trazem
`received_quantity` por célula, só por fornecedor (Seção 1). A coluna fica ausente aqui.

⚠️ **Validado em execução**: muitas linhas ficam com `Capacidade Hoje` nula. Isso é esperado —
a Seção 2 só traz células com `status IN ('available','approved','incubation')`, então células
que alocaram no passado (aparecem em `capacity_use`) mas hoje estão inativas/descontinuadas no
Muninn não têm capacidade vigente. Não é erro de join.


In [18]:
df_final_correto = df_capacidade_celula.merge(
    df_alocado_por_celula,
    on=["supplier_name", "cell_number", "cell_label"],
    how="outer",
)

tabela_final_correta = df_final_correto.rename(columns={
    "supplier_name": "Fornecedor",
    "cell_label": "Célula",
    "media_alocado_celula_12m": "Média de Alocado (Data Planejada) - 12 ÚLTIMOS MESES",
    "capacidade_mensal_hoje": "Capacidade Hoje",
})[[
    "Fornecedor",
    "Célula",
    "Média de Alocado (Data Planejada) - 12 ÚLTIMOS MESES",
    "n_meses_com_dado",
    "Capacidade Hoje",
]].sort_values("Fornecedor").reset_index(drop=True)

tabela_final_correta



,Fornecedor,Célula,Média de Alocado (Data Planejada) - 12 ÚLTIMOS MESES,n_meses_com_dado,Capacidade Hoje
0,ABBA,Polo Core,219.125,8,<NA>
1,ABBA,Henley Core,0.000,8,<NA>
2,ABBA,Core T-shirt,331.750,8,<NA>
3,ART LIVRE,modal/visup 1 - Gola U,9050.000,9,30000
4,ART LIVRE,modal/visup 2 - Gola V,1550.625,8,<NA>
...,...,...,...,...,...
195,WARUSKY,Vestido Sharp InLounge,0.000,4,8000
196,WARUSKY,Casaco Studio InLounge Feminino,0.000,2,<NA>
197,WARUSKY,Casaco Studio InLounge Masculino,1250.500,2,<NA>
198,WARUSKY,Minimal Top InLounge,3690.000,4,<NA>


## 3d. Query Única Consolidada (para rodar direto no BigQuery)

**Substitui a abordagem via `capacity_use`** (Seções 3b/3c) por um caminho melhor, validado em
2026-08-17:

- **Célula por OP**: `integrated.supply_production_orders.production_unit_cell_number`, join por
  `op_code` com `sop_silver.supply_chain_efficiency_model_input` (mesma tabela/filtros da
  Seção 1). Validado: **0 de 15.012 OPs têm mais de uma célula** (sem risco de fan-out) e
  **96,8% das linhas de eficiência resolvem célula** (66.183/68.383).
- **Média de Alocado** = `planned_quantity` por `dt_planned_entry_warehouse`.
- **Média de Produzida** = `received_quantity` por `dt_largest_entry_warehouse` (quantidade
  real que entrou no armazém — não existe `dt_received_entry_warehouse`, esse nome de coluna
  não existe na tabela e gera erro 400 no BigQuery; corrigido também na Seção 1).
- **Vantagem sobre `capacity_use`**: cobertura chega a **12 meses completos** (testado:
  `n_meses` até 12), contra o máximo de 9 meses de `capacity_use` (limitado pelo snapshot mais
  antigo em 2025-11-21).

⚠️ `Capacidade Hoje` continua vindo exclusivamente do cadastro Muninn (snapshot atual) —
não muda.

⚠️ **Coluna `cell_number` adicionada** (2026-08-17): o rótulo `celula` (`cell_label`) pode
colidir entre células distintas do mesmo fornecedor (ver achado na validação abaixo — caso
GOAT). `cell_number` é a chave numérica real e única; usar sempre `cell_number` para
agrupar/deduplicar, e `celula` só para exibição.

⚠️ **`capacidade_hoje` agregada por `(supplier_name, cell_number)`** (2026-08-17): o cadastro
Muninn tem grão `(supplier_name, cell_number, cell_label)` — um mesmo `cell_number` pode ter
mais de um label/produto cadastrado com capacidades diferentes (ex.: STILE COMERCIAL LTDA,
cell_number 1 = "Camiseta Algodao" + "Perfect top", capacidades 4000 e 12000). Isso causava
fan-out no join final (6 duplicatas encontradas em `(fornecedor, cell_number)`). Corrigido
somando a capacidade de todos os labels de um `cell_number` (a célula física produz todos
aqueles itens) e concatenando os labels em `cell_label` só para exibição — grão final agora é
de fato único em `(fornecedor, cell_number)`.


In [25]:
sql_tabela_final_consolidada = """
WITH capacidade_hoje_por_label AS (
    -- Capacidade vigente no cadastro Muninn (snapshot atual, nao historico).
    -- Grao real: (supplier_name, cell_number, cell_label) -- um cell_number pode ter
    -- mais de um label/produto cadastrado (ex.: STILE tem cell_number=1 com "Camiseta
    -- Algodao" e "Perfect top", capacidades diferentes). Por isso agregamos abaixo.
    SELECT
        s.alias AS supplier_name,
        ampu.apparel_manufacturer_cell_number AS cell_number,
        ampu.label AS cell_label,
        4 * MAX(ampup.weekly_maximum_productive_capacity) AS capacidade_mensal_hoje
    FROM `insider-data-lake.integrated.muninn_apparel_manufacturer_production_units_products` AS ampup
    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturers_products` AS amp
        ON amp.id = ampup.apparel_manufacturer_product_id
    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturer_production_units` AS ampu
        ON ampu.id = ampup.apparel_manufacturer_production_unit_id
    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturers` AS am
        ON am.id = ampu.apparel_manufacturer_id
    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS s
        ON s.id = am.supplier_id
    WHERE amp.status IN ('available', 'approved', 'incubation')
      AND ampup.weekly_maximum_productive_capacity > 0
    GROUP BY supplier_name, cell_number, cell_label
),
capacidade_hoje AS (
    -- Agregado ao grao real da tabela final: (supplier_name, cell_number).
    -- capacidade_mensal_hoje = soma das capacidades de todos os labels daquele
    -- cell_number (a celula fisica produz todos aqueles itens); cell_label vira
    -- a concatenacao dos labels, so para exibicao.
    SELECT
        supplier_name,
        cell_number,
        STRING_AGG(DISTINCT cell_label, ' / ' ORDER BY cell_label) AS cell_label,
        SUM(capacidade_mensal_hoje) AS capacidade_mensal_hoje
    FROM capacidade_hoje_por_label
    GROUP BY supplier_name, cell_number
),
op_cell AS (
    -- Celula por OP. Validado: 0 OPs com mais de 1 celula (sem fan-out).
    SELECT DISTINCT production_order_code AS op_code, production_unit_cell_number AS cell_number
    FROM `insider-data-lake.integrated.supply_production_orders`
    WHERE production_unit_cell_number IS NOT NULL
),
base AS (
    SELECT
        e.supplier_name,
        oc.cell_number,
        DATE_TRUNC(e.dt_planned_entry_warehouse, MONTH) AS mes_planejado,
        -- dt_largest_entry_warehouse = data real de entrada no armazem
        -- (dt_received_entry_warehouse NAO existe nesta tabela)
        DATE_TRUNC(e.dt_largest_entry_warehouse, MONTH) AS mes_recebido,
        e.planned_quantity,
        e.received_quantity
    FROM `insider-data-lake.sop_silver.supply_chain_efficiency_model_input` AS e
    LEFT JOIN op_cell AS oc ON oc.op_code = e.op_code
    WHERE e.production_order_type NOT IN ('flexible', 'converted')
      AND e.cycle_name IS NOT NULL
      AND (e.supplier_relationship_status IS NULL
           OR e.supplier_relationship_status NOT IN ('terminated', 'discontinued'))
),
alocado_mensal AS (
    SELECT supplier_name, cell_number, mes_planejado AS mes, SUM(planned_quantity) AS qtd_alocada
    FROM base
    WHERE mes_planejado >= DATE_TRUNC(DATE_SUB(CURRENT_DATE(), INTERVAL 12 MONTH), MONTH)
      AND mes_planejado < DATE_TRUNC(CURRENT_DATE(), MONTH)
    GROUP BY supplier_name, cell_number, mes
),
produzido_mensal AS (
    SELECT supplier_name, cell_number, mes_recebido AS mes, SUM(received_quantity) AS qtd_produzida
    FROM base
    WHERE mes_recebido >= DATE_TRUNC(DATE_SUB(CURRENT_DATE(), INTERVAL 12 MONTH), MONTH)
      AND mes_recebido < DATE_TRUNC(CURRENT_DATE(), MONTH)
    GROUP BY supplier_name, cell_number, mes
),
alocado_produzido_celula AS (
    SELECT
        COALESCE(a.supplier_name, p.supplier_name) AS supplier_name,
        COALESCE(a.cell_number, p.cell_number) AS cell_number,
        AVG(a.qtd_alocada) AS media_alocado_12m,
        AVG(p.qtd_produzida) AS media_produzida_12m,
        COUNT(DISTINCT a.mes) AS n_meses_alocado,
        COUNT(DISTINCT p.mes) AS n_meses_produzido
    FROM alocado_mensal AS a
    FULL OUTER JOIN produzido_mensal AS p
        ON a.supplier_name = p.supplier_name AND a.cell_number = p.cell_number AND a.mes = p.mes
    GROUP BY supplier_name, cell_number
)
SELECT
    COALESCE(ch.supplier_name, ap.supplier_name) AS fornecedor,
    COALESCE(ch.cell_number, ap.cell_number) AS cell_number,
    COALESCE(ch.cell_label, CAST(ap.cell_number AS STRING)) AS celula,
    ap.media_alocado_12m AS media_alocado_12_ultimos_meses,
    ap.media_produzida_12m AS media_produzida_12_ultimos_meses,
    ap.n_meses_alocado,
    ap.n_meses_produzido,
    ch.capacidade_mensal_hoje AS capacidade_hoje
FROM capacidade_hoje AS ch
FULL OUTER JOIN alocado_produzido_celula AS ap
    ON ch.supplier_name = ap.supplier_name
    AND ch.cell_number = ap.cell_number
ORDER BY fornecedor, cell_number
"""

df_tabela_final_consolidada = query_to_dataframe(sql_tabela_final_consolidada)
df_tabela_final_consolidada.head(10)



/Users/insider/LA_Coding_Projects/.venv-1/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,fornecedor,cell_number,celula,media_alocado_12_ultimos_meses,media_produzida_12_ultimos_meses,n_meses_alocado,n_meses_produzido,capacidade_hoje
0,ABBA,1,1,3367.333333,1903.333333,6,3,<NA>
1,ABBA,2,2,1514.000000,2538.000000,2,1,<NA>
2,ABBA,3,3,7504.166667,6043.400000,6,5,<NA>
3,ART LIVRE,1,modal/visup 1 - Gola U,42957.500000,37878.583333,12,12,30000
4,ART LIVRE,2,2,1638.000000,798.000000,2,3,<NA>
5,ART LIVRE,3,3,3933.000000,1359.800000,1,5,<NA>
6,ART LIVRE,4,4,2224.000000,2663.666667,2,3,<NA>
7,ARTIGO X,1,bone,4196.818182,3514.833333,11,12,10000
8,ASGA BRINDES,1,Brindes,NaN,NaN,<NA>,<NA>,20000
9,AZZURRA,1,Authentico/ Hydrid Jogger,2260.500000,1649.600000,4,5,2000


In [26]:
# Validacao do grao: (fornecedor, cell_number) deve ser unico — chave real
dup = df_tabela_final_consolidada.duplicated(subset=["fornecedor", "cell_number"], keep=False)
print("Total de linhas:", len(df_tabela_final_consolidada))
print("Linhas duplicadas em (fornecedor, cell_number):", dup.sum())
if dup.sum() > 0:
    display(df_tabela_final_consolidada[dup].sort_values(["fornecedor", "cell_number"]))

# Checagem cruzada: soma do alocado por celula deve bater com o alocado do fornecedor (Secao 1)
soma_por_fornecedor = (
    df_tabela_final_consolidada
    .groupby("fornecedor", as_index=False)["media_alocado_12_ultimos_meses"]
    .sum()
    .rename(columns={"media_alocado_12_ultimos_meses": "soma_alocado_celulas"})
)
comparacao = soma_por_fornecedor.merge(
    df_alocado_entrega[["supplier_name", "media_alocado_12m"]],
    left_on="fornecedor", right_on="supplier_name", how="inner",
)
comparacao["diferenca_pct"] = (
    (comparacao["soma_alocado_celulas"] - comparacao["media_alocado_12m"])
    / comparacao["media_alocado_12m"]
) * 100
comparacao.head(15)


Total de linhas: 188
Linhas duplicadas em (fornecedor, cell_number): 0


,fornecedor,soma_alocado_celulas,supplier_name,media_alocado_12m,diferenca_pct
0,ABBA,12385.500000,ABBA,11376.166667,8.872350
1,ART LIVRE,50752.500000,ART LIVRE,43928.916667,15.533238
2,ARTIGO X,4196.818182,ARTIGO X,4196.818182,0.000000
3,AZZURRA,7774.242857,AZZURRA,5484.875000,41.739654
4,BAE BRASIL,194141.416667,BAE BRASIL,186447.833333,4.126400
5,BITEX INDUSTRIA DE CONFECCOES LTDA,40.000000,BITEX INDUSTRIA DE CONFECCOES LTDA,40.000000,0.000000
6,BLUTEXTIL,12164.000000,BLUTEXTIL,9731.200000,25.000000
7,BLX TRADING,16495.000000,BLX TRADING,16495.000000,0.000000
8,BY COTTON,61866.246753,BY COTTON,55989.000000,10.497145
9,CLARA BELLA,11231.271429,CLARA BELLA,6668.888889,68.412934


In [22]:
# Investigar a causa da divergencia (ex.: DALOP, 155%)
display(df_tabela_final_consolidada[df_tabela_final_consolidada["fornecedor"] == "DALOP"])
display(df_tabela_final_consolidada[df_tabela_final_consolidada["fornecedor"] == "GOAT"])


,fornecedor,celula,media_alocado_12_ultimos_meses,media_produzida_12_ultimos_meses,n_meses_alocado,n_meses_produzido,capacidade_hoje
47,DALOP,NaN,823.750000,993.000000,4,1,<NA>
48,DALOP,1,NaN,1407.000000,0,1,<NA>
49,DALOP,11,911.500000,672.500000,4,2,<NA>
50,DALOP,12,921.333333,600.666667,3,3,<NA>
51,DALOP,Legging Dopamin,2712.666667,1456.000000,6,8,3000
52,DALOP,Regata,3041.571429,2836.285714,7,7,4000
53,DALOP,Saia Breeze,NaN,NaN,<NA>,<NA>,4000
54,DALOP,Shorts Fitness Dopamin Biker,1202.500000,1399.000000,2,3,2000
55,DALOP,Shorts Fitness Dopamin Curto,809.800000,1674.333333,5,3,2000
56,DALOP,Shorts Fitness Dopamin Midi,1019.000000,1116.800000,4,5,1600


,fornecedor,celula,media_alocado_12_ultimos_meses,media_produzida_12_ultimos_meses,n_meses_alocado,n_meses_produzido,capacidade_hoje
83,GOAT,NaN,NaN,304.00,0,1,<NA>
84,GOAT,3,134.0,108.00,1,1,<NA>
85,GOAT,4,2296.5,739.00,2,2,<NA>
86,GOAT,Structure / High Neck,857.4,1012.75,5,4,1000
87,GOAT,Structure Cropped,474.0,875.50,2,2,3000
88,GOAT,boucle,1064.7,1205.00,10,9,1400
89,GOAT,boucle,3024.4,2517.25,10,12,2500


### ⚠️ Achado de validação (2026-08-17): grão OK, mas soma por célula ≠ média do fornecedor

**Grão confirmado (após correção)**: a chave de join real é `(supplier_name, cell_number)` —
íntegra, **0 duplicatas** confirmadas em execução. Duas causas de fan-out foram encontradas e
corrigidas nesta mesma validação:

1. **Rótulo (`cell_label`) pode colidir** entre células distintas (ex.: GOAT tinha duas células
   diferentes rotuladas "boucle", `cell_number` diferente). Corrigido adicionando `cell_number`
   como coluna explícita e chave real de deduplicação/agrupamento — `celula` (label) é só para
   exibição.
2. **Um `cell_number` pode ter mais de um `cell_label`/produto cadastrado no Muninn** (ex.:
   STILE COMERCIAL LTDA, cell_number 1 = "Camiseta Algodao" + "Perfect top", capacidades 4000 e
   12000). Corrigido agregando `capacidade_hoje` para o grão `(supplier_name, cell_number)`
   antes do join final — soma as capacidades dos labels daquele cell_number e concatena os
   labels em `cell_label` só para exibição.

**Divergência real encontrada (aceita, sem correção — ponto 2 do usuário)**: somar
`media_alocado_12_ultimos_meses` de todas as células de um fornecedor **não bate** com a média
do fornecedor calculada na Seção 1 (chega a +155% no caso da DALOP). Causa raiz: cada célula tem
um `n_meses_alocado` diferente (ex.: GOAT tem células com 1, 2, 5, 10 meses de atividade dentro
da janela de 12 meses). Como a média por célula divide pelo número de meses **daquela célula**,
células com poucos meses ativos ficam com média inflada, e a soma dessas médias não reconstrói
o total do fornecedor.

**Implicação prática**: os valores por célula individualmente estão corretos (média real de cada
célula nos meses em que ela operou), mas **não somar** para comparar com o total do fornecedor
sem antes normalizar pelo mesmo denominador de meses (ex.: dividir sempre por 12, não por
`n_meses`, se o objetivo for reconciliar com o agregado do fornecedor).


## 4. Export (opcional)

In [20]:
today = pd.to_datetime("today").strftime("%Y%m%d")
output_path = "../../outputs/capacidade_producao/"

# import os
# os.makedirs(output_path, exist_ok=True)
# tabela_final.to_csv(f"{output_path}capacidade_fornecedor_celula_{today}.csv", index=False)